# [BRONZE] Ingestion — PROMISE + Red Hat Jira

**Layer:** Bronze (raw, immutable)
**Author:** ShiftMetrics Analytics — EAFIT SI7006 Trabajo 3

## Sources
| Source | Path | Files | Schema |
|---|---|---|---|
| PROMISE Defect | `landing/promise/<project>/<project>-<version>.csv` | 41 | 22 cols (C&K + Halstead + McCabe) |
| Red Hat Public Jira | `landing/redhat/<system>.csv` | 250 | 9 cols (Jira issue lifecycle) |

## Sinks (Delta)
- `workspace.shiftmetrics_bronze.code_metrics_raw`     — partitioned by `_project`
- `workspace.shiftmetrics_bronze.process_issues_raw`   — partitioned by `_system_bucket`

## Audit columns
`_ingestion_ts`, `_source_file`, `_project`/`_system`, `_version` (PROMISE only)

> **Note:** Unity Catalog forbids `input_file_name()`. We use the modern
> `_metadata.file_path` API, which is also strictly typed and richer.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

VOLUME_LANDING = "/Volumes/workspace/shiftmetrics_bronze/lakehouse_vol/landing"
BRONZE         = "workspace.shiftmetrics_bronze"

run_started = datetime.utcnow().isoformat() + "Z"
print(f"-> Bronze ingestion started: {run_started}")
print(f"  Spark version: {spark.version}")
print(f"  Catalog:       {spark.catalog.currentCatalog()}")

-> Bronze ingestion started: 2026-05-10T15:43:07.583229Z
  Spark version: 4.1.0
  Catalog:       workspace


## [1] PROMISE — Code metrics (41 files → 1 Delta table)

In [0]:
promise_schema = StructType([
    StructField("name",    StringType(),  False),
    StructField("wmc",     DoubleType(),  True),
    StructField("dit",     DoubleType(),  True),
    StructField("noc",     DoubleType(),  True),
    StructField("cbo",     DoubleType(),  True),
    StructField("rfc",     DoubleType(),  True),
    StructField("lcom",    DoubleType(),  True),
    StructField("ca",      DoubleType(),  True),
    StructField("ce",      DoubleType(),  True),
    StructField("npm",     DoubleType(),  True),
    StructField("lcom3",   DoubleType(),  True),
    StructField("loc",     DoubleType(),  True),
    StructField("dam",     DoubleType(),  True),
    StructField("moa",     DoubleType(),  True),
    StructField("mfa",     DoubleType(),  True),
    StructField("cam",     DoubleType(),  True),
    StructField("ic",      DoubleType(),  True),
    StructField("cbm",     DoubleType(),  True),
    StructField("amc",     DoubleType(),  True),
    StructField("max_cc",  DoubleType(),  True),
    StructField("avg_cc",  DoubleType(),  True),
    StructField("bug",     IntegerType(), True),
])

# Note: select("*", "_metadata.file_path") replaces the legacy input_file_name()
promise_df = (
    spark.read
        .schema(promise_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .csv(f"{VOLUME_LANDING}/promise/*/*.csv")
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
        .withColumn("_ingestion_ts", F.current_timestamp())
        # Path: .../landing/promise/<project>/<project>-<version>.csv
        .withColumn("_project",
                    F.regexp_extract("_source_file", r"/promise/([^/]+)/", 1))
        .withColumn("_version",
                    F.regexp_extract("_source_file", r"-([0-9][0-9.a-z]*)\.csv$", 1))
)

promise_count = promise_df.count()
print(f"  PROMISE rows: {promise_count:,}")
promise_df.printSchema()

  PROMISE rows: 15,775
root
 |-- name: string (nullable = true)
 |-- wmc: double (nullable = true)
 |-- dit: double (nullable = true)
 |-- noc: double (nullable = true)
 |-- cbo: double (nullable = true)
 |-- rfc: double (nullable = true)
 |-- lcom: double (nullable = true)
 |-- ca: double (nullable = true)
 |-- ce: double (nullable = true)
 |-- npm: double (nullable = true)
 |-- lcom3: double (nullable = true)
 |-- loc: double (nullable = true)
 |-- dam: double (nullable = true)
 |-- moa: double (nullable = true)
 |-- mfa: double (nullable = true)
 |-- cam: double (nullable = true)
 |-- ic: double (nullable = true)
 |-- cbm: double (nullable = true)
 |-- amc: double (nullable = true)
 |-- max_cc: double (nullable = true)
 |-- avg_cc: double (nullable = true)
 |-- bug: integer (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingestion_ts: timestamp (nullable = false)
 |-- _project: string (nullable = false)
 |-- _version: string (nullable = false)



In [0]:
(
    promise_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("_project")
        .saveAsTable(f"{BRONZE}.code_metrics_raw")
)

spark.sql(f"""
    ALTER TABLE {BRONZE}.code_metrics_raw
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = true,
        'delta.autoOptimize.autoCompact'   = true
    )
""")

spark.sql(f"OPTIMIZE {BRONZE}.code_metrics_raw ZORDER BY (_version)")

spark.sql(f"""
    COMMENT ON TABLE {BRONZE}.code_metrics_raw IS
    'PROMISE defect dataset — C&K + Halstead + McCabe code metrics for 11 Apache projects across multiple versions. Target: bug (defect count, binarizable). Partitioned by _project, Z-Ordered by _version.'
""")

print(f"  [OK] Wrote {BRONZE}.code_metrics_raw")

display(spark.sql(f"""
    SELECT _project,
           COUNT(*)                AS rows,
           COUNT(DISTINCT _version) AS versions,
           SUM(CASE WHEN bug > 0 THEN 1 ELSE 0 END) AS buggy_modules,
           ROUND(100.0 * SUM(CASE WHEN bug > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_buggy
    FROM {BRONZE}.code_metrics_raw
    GROUP BY _project
    ORDER BY _project
"""))

  [OK] Wrote workspace.shiftmetrics_bronze.code_metrics_raw


_project,rows,versions,buggy_modules,pct_buggy
ant,1692,5,350,20.7
camel,2784,4,562,20.2
ivy,704,3,119,16.9
jedit,1749,5,303,17.3
log4j,449,3,260,57.9
lucene,782,3,438,56.0
poi,1378,4,707,51.3
synapse,635,3,162,25.5
velocity,639,3,367,57.4
xalan,3320,4,1806,54.4


## [2] Red Hat — Process issues (250 files → 1 Delta table)

In [0]:
# Schema validated in FASE 5 against 250 files (all match this header)
redhat_raw = (
    spark.read
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .csv(f"{VOLUME_LANDING}/redhat/*.csv")
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
)

# Normalize column names → snake_case
column_map = {
    "Issue key":     "issue_key",
    "Issue Type":    "issue_type",
    "Status":        "status",
    "Project key":   "project_key",
    "Project name":  "project_name",
    "Project type":  "project_type",
    "Resolution":    "resolution",
    "Created":       "created_str",
    "Resolved":      "resolved_str",
}
for src, tgt in column_map.items():
    if src in redhat_raw.columns:
        redhat_raw = redhat_raw.withColumnRenamed(src, tgt)

# Cast dates: format dd/MM/yyyy HH:mm  (e.g. "16/06/2022 23:13")
DATE_FMT = "dd/MM/yyyy HH:mm"

redhat_df = (
    redhat_raw
        .withColumn("created",  F.to_timestamp("created_str",  DATE_FMT))
        .withColumn("resolved", F.to_timestamp("resolved_str", DATE_FMT))
        .drop("created_str", "resolved_str")
        .withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_system",
                    F.regexp_extract("_source_file", r"/redhat/([^/]+)\.csv$", 1))
        # Bucket by first letter for partition pruning (250 distinct systems → too many)
        .withColumn("_system_bucket",
                    F.upper(F.substring("project_key", 1, 1)))
)

redhat_count = redhat_df.count()
print(f"  Red Hat rows: {redhat_count:,}")
redhat_df.printSchema()

  Red Hat rows: 504,640
root
 |-- issue_key: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- project_key: string (nullable = true)
 |-- project_name: string (nullable = true)
 |-- project_type: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- created: timestamp (nullable = true)
 |-- resolved: timestamp (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = false)
 |-- _system: string (nullable = false)
 |-- _system_bucket: string (nullable = true)



In [0]:
(
    redhat_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("_system_bucket")
        .saveAsTable(f"{BRONZE}.process_issues_raw")
)

spark.sql(f"""
    ALTER TABLE {BRONZE}.process_issues_raw
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = true,
        'delta.autoOptimize.autoCompact'   = true
    )
""")

spark.sql(f"OPTIMIZE {BRONZE}.process_issues_raw ZORDER BY (project_key, status)")

spark.sql(f"""
    COMMENT ON TABLE {BRONZE}.process_issues_raw IS
    'Red Hat Public Jira PBIs 2001-2024 — process metrics raw. 250 systems × ~490k issues. Excludes RHD system (schema outlier, see DD-002). Partitioned by _system_bucket (first-letter), Z-Ordered by project_key + status.'
""")

print(f"  [OK] Wrote {BRONZE}.process_issues_raw")

display(spark.sql(f"""
    SELECT issue_type,
           COUNT(*) AS issues,
           ROUND(AVG(DATEDIFF(resolved, created)), 1) AS avg_days_to_resolve
    FROM {BRONZE}.process_issues_raw
    GROUP BY issue_type
    ORDER BY issues DESC
"""))

  [OK] Wrote workspace.shiftmetrics_bronze.process_issues_raw


issue_type,issues,avg_days_to_resolve
Bug,223824,146.9
Task,112917,114.2
Story,71106,143.2
Feature Request,33654,333.5
Enhancement,24826,237.6
Epic,13247,283.5
Component Upgrade,9650,43.5
Feature,6656,231.2
Spike,3013,109.1
Release,1104,69.1


## [3] Bronze layer summary

In [0]:
display(spark.sql(f"""
    SELECT table_name,
           table_type,
           comment
    FROM workspace.information_schema.tables
    WHERE table_schema = 'shiftmetrics_bronze'
      AND table_type   = 'MANAGED'
    ORDER BY table_name
"""))

print(f"\n-> Bronze ingestion finished at {datetime.utcnow().isoformat()}Z")
print(f"  PROMISE rows ingested: {promise_count:,}")
print(f"  Red Hat rows ingested: {redhat_count:,}")
print(f"  Total rows in Bronze:  {promise_count + redhat_count:,}")

table_name,table_type,comment
code_metrics_raw,MANAGED,"PROMISE defect dataset — C&K + Halstead + McCabe code metrics for 11 Apache projects across multiple versions. Target: bug (defect count, binarizable). Partitioned by _project, Z-Ordered by _version."
process_issues_raw,MANAGED,"Red Hat Public Jira PBIs 2001-2024 — process metrics raw. 250 systems × ~490k issues. Excludes RHD system (schema outlier, see DD-002). Partitioned by _system_bucket (first-letter), Z-Ordered by project_key + status."



-> Bronze ingestion finished at 2026-05-10T15:43:45.429214Z
  PROMISE rows ingested: 15,775
  Red Hat rows ingested: 504,640
  Total rows in Bronze:  520,415


In [0]:
# ============================================================================
# DIAGNOSTIC — Validate row counts file-by-file vs Delta table
# ============================================================================

# 1. Count actual rows in CSV files (ground truth via Spark text reader)
csv_counts = (
    spark.read
        .option("header", "false")
        .text(f"{VOLUME_LANDING}/promise/*/*.csv")
        .select("*", F.col("_metadata.file_path").alias("file_path"))
        .groupBy("file_path")
        .count()
        .withColumn("data_rows", F.col("count") - 1)   # subtract header
        .withColumn("project",
                    F.regexp_extract("file_path", r"/promise/([^/]+)/", 1))
        .withColumn("version",
                    F.regexp_extract("file_path", r"-([0-9][0-9.a-z]*)\.csv$", 1))
        .select("project", "version", "data_rows")
        .orderBy("project", "version")
)

# 2. Count rows in the Delta table per (project, version)
delta_counts = (
    spark.table(f"{BRONZE}.code_metrics_raw")
        .groupBy("_project", "_version")
        .count()
        .withColumnRenamed("_project", "project")
        .withColumnRenamed("_version", "version")
        .withColumnRenamed("count", "delta_rows")
)

# 3. Diff
diff = (
    csv_counts.join(delta_counts, ["project", "version"], "left")
        .fillna({"delta_rows": 0})
        .withColumn("missing", F.col("data_rows") - F.col("delta_rows"))
        .orderBy(F.desc("missing"))
)

print("▶ Files with missing rows after ingestion (top 15):")
display(diff.filter(F.col("missing") != 0))

print("▶ Totals:")
totals = diff.agg(
    F.sum("data_rows").alias("csv_total"),
    F.sum("delta_rows").alias("delta_total"),
    F.sum("missing").alias("total_missing"),
)
display(totals)

totals.show(truncate=False)

▶ Files with missing rows after ingestion (top 15):


project,version,data_rows,delta_rows,missing


▶ Totals:


csv_total,delta_total,total_missing
15775,15775,0


+---------+-----------+-------------+
|csv_total|delta_total|total_missing|
+---------+-----------+-------------+
|15775    |15775      |0            |
+---------+-----------+-------------+

